# 4차시 ― 데이터 분석 실습 ② · Netflix 데이터로 진짜 분석하기

> 바로 앞 노트북(`s4-01`)에서 **pandas·matplotlib·seaborn**의 기본기를 익혔습니다. 이번에는 그 도구로 **진짜 데이터셋**을 분석합니다. 넷플릭스에 올라온 약 8,800편의 영화·드라마 정보가 담긴 표를 불러와, 질문을 던지고 코드로 답을 찾아봅니다.
>
> 3차시(`s3-03`)에서 **순수 파이썬**(리스트·딕셔너리·반복문)만으로 표 데이터를 다뤘던 것을 기억하시나요? 합계 하나, 그룹별 집계 하나에도 여러 줄이 필요했습니다. 같은 일을 pandas로 하면 **대부분 한 줄**입니다. 그 차이를 직접 확인하는 것이 이 노트북의 또 다른 목표입니다.
>
> **진행 방식**: 각 분석은 **질문 → pandas 코드 → 결과 해석** 순서로 이어집니다. 코드 셀을 클릭하고 `Shift + Enter` 로 위에서 아래로 실행하세요. 앞 셀에서 만든 `df` 를 뒤 셀에서 계속 사용합니다.

**다루는 내용**
1. 데이터 불러오기와 구조 파악 (`read_csv`, `shape`, `head`, `info`, `isna`)
2. 영화 vs 드라마 — 무엇이 더 많을까? (`value_counts`)
3. 콘텐츠가 가장 많은 국가는? (결측치 처리)
4. 최근에 추가된 콘텐츠 (날짜 변환·정렬)
5. 조건으로 데이터 거르기 (필터링, 불리언 인덱싱)
6. 영화 상영시간 분포 (문자열 전처리·히스토그램)
7. 넷플릭스가 가장 바쁜 달은? (월별 추이)
8. 연도별 콘텐츠 증가 추세 (`groupby`)
9. 가장 많은 작품을 만든 감독 Top 10

데이터 출처: Kaggle ― *Netflix Movies and TV Shows*

> 📌 **✏️ 표시가 붙은 "연습문제"는 직접 풀어 보는 칸입니다.** 먼저 스스로 작성해 본 뒤 정답을 확인하세요.

## 0. 준비 — 라이브러리와 한글 폰트

분석을 시작하기 전에 라이브러리를 불러오고, 그래프에서 한글이 깨지지 않도록 폰트를 설정합니다. 아래 두 셀은 `s4-01` 에서 했던 것과 같습니다.

> **Colab 사용자**: 첫 번째 셀(폰트 설치)을 실행한 뒤 **[런타임] → [런타임 다시 시작]** 을 한 번 눌러 주세요. 그래야 한글 폰트가 적용됩니다. 재시작 후에는 폰트 설치 셀을 건너뛰고 두 번째 셀부터 실행하면 됩니다.

In [1]:
# Google Colab 환경: 한글 폰트 설치 (한 번만 실행)
# 로컬 Jupyter 에서 한글 폰트가 이미 있다면 이 셀은 건너뛰어도 됩니다.
!apt-get install -y fonts-nanum -qq
!fc-cache -fv

print("폰트 설치 완료. Colab이라면 [런타임] > [런타임 다시 시작] 후 아래 셀부터 실행하세요.")

Selecting previously unselected package fonts-nanum.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 12 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no such directory
/root/.fonts: skipping, no such directory
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/humor-sans: skipping, looped directory det

In [1]:
# 라이브러리 불러오기
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# 한글 폰트 설정 (그래프에서 한글이 깨지지 않도록)
import matplotlib.font_manager as fm
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
try:
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
except FileNotFoundError:
    pass  # 폰트가 없으면 기본 폰트로 진행 (그래프 한글만 깨질 수 있음)
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호 깨짐 방지
sns.set_style('whitegrid')

print("라이브러리 준비 완료")

라이브러리 준비 완료


## 1. 데이터 불러오기

실습 데이터(`netflix_titles.csv`)는 강의 저장소에 들어 있습니다. 아래 셀로 저장소를 내려받습니다. **한 번만** 실행하면 됩니다.

In [2]:
# 강의 저장소 clone (한 번만 실행)
!git clone https://github.com/chanmuzi/hknu-ai-2026-summer.git

Cloning into 'hknu-ai-2026-summer'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (343/343), done.
remote: Compressing objects: 100% (230/230), done.
remote: Total 343 (delta 235), reused 220 (delta 113), pack-reused 0 (from 0)
Receiving objects: 100% (343/343), 9.12 MiB | 26.23 MiB/s, done.
Resolving deltas: 100% (235/235), done.


내려받은 CSV 파일을 pandas 의 `read_csv` 로 읽어 **DataFrame**(표) 으로 만듭니다. 변수 이름은 관례적으로 `df`(data frame) 를 씁니다.

In [3]:
# CSV 파일을 DataFrame 으로 읽기
df = pd.read_csv('hknu-ai-2026-summer/session-4-data-analysis/data/netflix_titles.csv')

# Colab 에서 /content 경로를 쓰려면 아래처럼 절대경로로 적어도 됩니다.
# df = pd.read_csv('/content/hknu-ai-2026-summer/session-4-data-analysis/data/netflix_titles.csv')

df.head()  # 앞에서 5행 미리보기

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## 2. 데이터 구조 파악하기

분석을 시작하기 전에 **이 표가 어떻게 생겼는지** 부터 살펴봅니다. 행은 콘텐츠 하나, 열은 그 콘텐츠의 속성(제목·종류·국가·추가일 등)입니다.

| 메서드 | 알려주는 것 |
|---|---|
| `df.shape` | (행 개수, 열 개수) |
| `df.head(n)` | 앞에서 n 개 행 |
| `df.columns` | 열 이름 목록 |
| `df.info()` | 열별 자료형과 비어 있지 않은 값의 개수 |
| `df.isna().sum()` | 열별 결측치(빈 값) 개수 |

In [ ]:
# 행·열 개수 (콘텐츠 수, 속성 수)
print('행, 열 개수:', df.shape)
print('열 이름:', list(df.columns))

In [ ]:
# 열별 자료형과 비어 있지 않은 값의 개수
df.info()

`info()` 출력에서 `Non-Null Count` 가 전체 행 수보다 작은 열은 **빈 값(결측치)이 있다**는 뜻입니다. 결측치 개수만 따로 세어 봅니다.

In [ ]:
# 열별 결측치(빈 값) 개수 — 큰 순서로 정렬
df.isna().sum().sort_values(ascending=False)

> **결과 해석**: `director`(감독), `cast`(출연), `country`(국가) 에 빈 값이 많습니다. 분석할 때 이 열을 쓰려면 **빈 값을 먼저 걸러내야** 결과가 정확합니다. 아래 분석들에서 `dropna(...)` 가 자주 등장하는 이유입니다.

## 3. 영화가 많을까, 드라마가 많을까?

**질문**: 넷플릭스에는 영화(Movie)와 TV 드라마(TV Show) 중 무엇이 더 많을까요?

`type` 열의 값마다 개수를 세면 됩니다. 3차시에서는 딕셔너리에 `get(키, 0) + 1` 로 직접 셌지만, pandas 에서는 **`value_counts()` 한 줄**입니다.

In [ ]:
# type 열의 값별 개수 세기
type_counts = df['type'].value_counts()
print(type_counts)

In [ ]:
# 막대그래프로 시각화
type_counts.plot(kind='bar', color=['#4C72B0', '#DD8452'], rot=0)
plt.title('콘텐츠 종류별 개수')
plt.ylabel('개수')
plt.show()

> **결과 해석**: 넷플릭스는 **영화(Movie)가 드라마(TV Show)보다 약 2배 많습니다.** 흔히 '넷플릭스 드라마'를 떠올리지만, 보유 편수로는 영화가 훨씬 많다는 점이 데이터로 확인됩니다.

## 4. 콘텐츠가 가장 많은 국가는?

**질문**: 어느 나라의 콘텐츠가 가장 많을까요?

`value_counts()` 는 결측치(NaN)를 **자동으로 빼고** 세므로, 따로 처리하지 않아도 됩니다. 상위 10개국만 `head(10)` 으로 봅니다.

In [ ]:
# 국가별 콘텐츠 수 상위 10개
top_countries = df['country'].value_counts().head(10)
print(top_countries)

In [ ]:
# 가로 막대그래프 (국가 이름이 길어 가로가 보기 편함)
top_countries.sort_values().plot(kind='barh', color='#4C72B0')
plt.title('콘텐츠 보유 상위 10개국')
plt.xlabel('개수')
plt.show()

> **결과 해석**: **미국**이 압도적 1위이고, **인도·영국**이 뒤를 잇습니다. 한 콘텐츠에 `United States, Canada` 처럼 여러 나라가 함께 적힌 경우는 별개 항목으로 세어지므로, 위 숫자는 '단독 제작 국가' 기준에 가깝다는 점을 감안해 읽습니다.

## 5. 가장 최근에 추가된 콘텐츠는?

**질문**: 넷플릭스에 가장 최근 올라온 작품은 무엇일까요?

`date_added` 열은 `September 9, 2021` 같은 **문자열**이라, 그대로 정렬하면 글자 순으로 정렬되어 엉뚱한 결과가 나옵니다. `pd.to_datetime` 으로 **날짜 자료형**으로 바꾼 뒤 정렬해야 합니다. (변환이 안 되는 값은 `errors='coerce'` 로 빈 날짜 처리)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">⚠️ 흔한 실수 ② · 날짜를 '문자열'인 채로 정렬 — 결과를 예측해 보세요</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>상황</b> &nbsp; 바로 아래 셀에서 <code>date_added</code> 를 날짜형으로 바꿉니다. 그 <b>전에</b>, 문자열 상태로 <code>max()</code>(가장 큰 값)를 구하면?<br><b>예측</b> &nbsp; '가장 최근 날짜'가 제대로 나올까요?
</div>
</div>

In [ ]:
# ⚠️ 흔한 실수 — 날짜가 아직 '문자열'이면 max()가 '사전순' 최대를 준다
print('문자열 상태 max:', df['date_added'].dropna().str.strip().max())
# → 'September ...' 같은 값이 나온다. 진짜 최신이 아니라 ㄱㄴㄷ(알파벳) 순서!

<details>
<summary><b>🔑 왜 이렇게 되나 · 올바른 코드</b></summary>

문자열의 <code>max()</code> 는 날짜 크기가 아니라 <b>글자 순서(사전순)</b>로 비교합니다. 그래서 <code>'September'</code> 가 <code>'December'</code> 보다 뒤로 가는 등 엉뚱한 결과가 나와요. <b>날짜는 반드시 <code>pd.to_datetime</code> 으로 바꾼 뒤</b> 비교·정렬해야 합니다 (그게 바로 아래 셀에서 하는 일).

```python
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
print('진짜 최신:', df['date_added'].max())   # 이제 날짜 크기로 비교된다
```

👉 <b>교훈</b> : 날짜·숫자는 <b>자료형부터</b> 확인. 문자열이면 정렬·min·max가 전부 틀립니다.

</details>

In [ ]:
# 문자열 -> 날짜 자료형으로 변환
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# 추가일 기준 내림차순 정렬 후 상위 5개의 일부 열만 보기
recent = df.sort_values('date_added', ascending=False)
recent[['title', 'type', 'country', 'date_added']].head()

> **결과 해석**: `sort_values('date_added', ascending=False)` 로 **가장 나중 날짜가 맨 위**로 옵니다. 원하는 열만 `[['title', ...]]` 로 골라 보면 결과가 훨씬 깔끔합니다.

## 6. 조건으로 데이터 거르기 — 한국 콘텐츠만

**질문**: 한국에서 제작된 콘텐츠만 따로 보고 싶다면?

`df[조건]` 형태로 **불리언 인덱싱**을 씁니다. 3차시에서 리스트 컴프리헨션 `[m for m in movies if ...]` 으로 걸렀던 것을, pandas 에서는 `df[df['country'] == 'South Korea']` 한 줄로 합니다.

In [ ]:
# country 가 'South Korea' 인 행만 골라내기
korea_df = df[df['country'] == 'South Korea']

print(f"한국 단독 제작 콘텐츠 수: {len(korea_df)}")
print()
print('한국 콘텐츠의 종류별 비율:')
print(korea_df['type'].value_counts())

조건을 **두 개 이상** 결합할 수도 있습니다. 각 조건을 괄호로 감싸고 `&`(그리고) 또는 `|`(또는) 로 잇습니다. 아래는 *한국에서 제작된 드라마(TV Show)* 만 거르는 예입니다.

In [ ]:
# 복합 조건: 한국 & 드라마
korea_show = df[(df['country'] == 'South Korea') & (df['type'] == 'TV Show')]
print(f"한국 드라마 수: {len(korea_show)}")
korea_show[['title', 'release_year']].head()

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">⚠️ 흔한 실수 ① · 값의 대소문자를 틀리면 — 결과를 예측해 보세요</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>상황</b> &nbsp; 영화만 고르려고 <code>df[df['type'] == 'movie']</code> 라고 썼습니다. (소문자 <code>movie</code>)<br><b>예측</b> &nbsp; 몇 건이 나올까요? 에러가 날까요, 아니면 숫자가 나올까요?
</div>
</div>

In [ ]:
# ⚠️ 흔한 실수 — 실제 값은 'Movie'(대문자 M)인데 'movie'로 필터
wrong = df[df['type'] == 'movie']
print('필터 결과:', len(wrong), '건   ← 넷플릭스에 영화가 하나도 없다고?!')
print('type 열의 실제 값:', df['type'].unique())

<details>
<summary><b>🔑 왜 이렇게 되나 · 올바른 코드</b></summary>

문자열 비교는 <b>대소문자를 정확히</b> 가립니다. 실제 값은 <code>'Movie'</code> 라서 <code>'movie'</code> 와는 하나도 안 맞아 <b>에러 없이 0건</b>이 나옵니다. <b>에러가 안 나서 더 위험한</b> 실수예요.

```python
df['type'].unique()          # ① 먼저 실제 값이 뭔지 확인하는 습관
movies = df[df['type'] == 'Movie']   # ② 정확한 값으로 필터
```

👉 <b>교훈</b> : 필터가 <b>0건</b>이면 데이터가 없는 게 아니라 <b>값을 잘못 적었을</b> 확률이 높습니다. <code>.unique()</code> 로 먼저 확인하세요.

</details>

> **결과 해석**: 미국 등 다른 나라와 달리, **한국은 영화보다 드라마(TV Show)의 비중이 높게** 나타납니다. 'K-드라마' 강세가 데이터에도 드러납니다.

## 7. 넷플릭스 영화는 보통 몇 분일까?

**질문**: 영화 한 편의 평균 상영시간과 분포는?

`duration` 열은 영화면 `90 min`, 드라마면 `2 Seasons` 처럼 **단위가 섞인 문자열**입니다. 영화만 골라 `' min'` 글자를 떼고 숫자로 바꿔야 평균을 낼 수 있습니다. 이 **전처리** 과정이 실제 분석에서 가장 자주 하는 일입니다.

In [ ]:
# 1) 영화 데이터만 추출 (.copy() 로 원본과 분리)
movies = df[df['type'] == 'Movie'].copy()

# 2) ' min' 글자를 떼고 숫자(정수)로 변환
movies['duration_min'] = movies['duration'].str.replace(' min', '', regex=False)
movies = movies.dropna(subset=['duration_min'])          # 변환 불가한 빈 값 제거
movies['duration_min'] = movies['duration_min'].astype(int)

# 3) 기초 통계 확인
print(f"평균 상영시간: {movies['duration_min'].mean():.1f} 분")
print(f"가장 짧은 영화: {movies['duration_min'].min()} 분")
print(f"가장 긴 영화: {movies['duration_min'].max()} 분")

In [ ]:
# 상영시간 분포를 히스토그램으로 (seaborn)
sns.histplot(movies['duration_min'], bins=30, kde=True, color='#4C72B0')
plt.title('넷플릭스 영화 상영시간 분포')
plt.xlabel('상영시간(분)')
plt.ylabel('영화 수')
plt.show()

> **결과 해석**: 대부분의 영화가 **90~100분 부근**에 몰려 있는 종 모양 분포입니다. 평균은 약 100분 안팎으로, 우리가 체감하는 '보통 영화 길이'와 일치합니다. `min`/`max` 를 보면 몇 분짜리 단편부터 수백 분짜리까지 폭이 넓다는 것도 알 수 있습니다.

## 8. 넷플릭스가 가장 바쁜 달은?

**질문**: 콘텐츠가 가장 많이 추가되는 달(월)은 언제일까요?

앞에서 날짜로 바꾼 `date_added` 에서 **월 정보만** 꺼냅니다. 날짜 자료형은 `.dt.month` (숫자 1~12), `.dt.month_name()` (영문 월 이름) 처럼 **`.dt` 접근자**로 연·월·요일 등을 쉽게 추출합니다.

In [ ]:
# 날짜가 비어 있는 행은 제외하고, 월(숫자)만 추출
dated = df.dropna(subset=['date_added']).copy()
dated['month'] = dated['date_added'].dt.month

# 1~12월 순서대로 개수 세기
monthly = dated['month'].value_counts().sort_index()
print(monthly)

In [ ]:
# 월별 추가량을 선그래프로
monthly.plot(kind='line', marker='o', color='#C44E52')
plt.title('월별 콘텐츠 추가량')
plt.xlabel('월')
plt.ylabel('추가된 콘텐츠 수')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.show()

> **결과 해석**: 연말·연초(특히 **7월, 12월** 부근)에 추가량이 상대적으로 많은 편입니다. 신규 시즌·연말 라인업이 몰리는 시기로 해석할 수 있습니다. `value_counts()` 뒤에 `.sort_index()` 를 붙여 **1→12월 순서**로 정렬한 점에 주목하세요(안 그러면 개수 많은 순으로 섞입니다).

## 9. 콘텐츠는 해마다 얼마나 늘었을까? — `groupby`

**질문**: 제작 연도(`release_year`) 별로 콘텐츠 수가 어떻게 변해 왔을까요?

여기서는 **`groupby`** 를 씁니다. `groupby('release_year')` 는 같은 연도끼리 묶고, 그 뒤 `.size()`(개수)나 `.mean()`(평균) 같은 집계를 이어 붙입니다. 3차시에서 `setdefault` 로 빈 리스트를 만들어 한 줄씩 모았던 그룹 집계를, pandas 는 **`groupby` 한 줄**로 끝냅니다.

In [ ]:
# 연도별 콘텐츠 개수 (2000년 이후만 보기)
yearly = df.groupby('release_year').size()
yearly_recent = yearly[yearly.index >= 2000]

yearly_recent.plot(kind='area', color='#4C72B0', alpha=0.5)
plt.title('제작 연도별 콘텐츠 수 (2000년 이후)')
plt.xlabel('제작 연도')
plt.ylabel('콘텐츠 수')
plt.show()

In [ ]:
# groupby 는 종류(type)별로도 한 번에 집계할 수 있습니다.
# 연도 x 종류 별 개수 -> 표(피벗) 형태로
by_year_type = df.groupby(['release_year', 'type']).size().unstack(fill_value=0)
by_year_type.tail(10)   # 최근 10개 연도

> **결과 해석**: 콘텐츠 수는 **2010년대 중후반에 급격히 증가**합니다(넷플릭스의 오리지널·글로벌 확장 시기와 맞물립니다). `groupby(['release_year','type'])` 처럼 **두 기준으로 묶고** `unstack` 으로 펼치면, 연도별 영화/드라마 추세를 한 표로 비교할 수 있습니다.

## 10. 가장 많은 작품을 만든 감독 Top 10

**질문**: 넷플릭스에 작품이 가장 많은 감독은 누구일까요?

`director` 열은 빈 값이 많으므로(2번에서 확인) **먼저 `dropna` 로 걸러낸 뒤** 세어야 합니다.

In [ ]:
# 감독이 비어 있는 행 제외 후 상위 10명
top_directors = df.dropna(subset=['director'])['director'].value_counts().head(10)
print(top_directors)

In [ ]:
# 가로 막대그래프로 시각화
top_directors.sort_values().plot(kind='barh', color='#4C72B0')
plt.title('작품 수 상위 10명 감독')
plt.xlabel('작품 수')
plt.tight_layout()
plt.show()

> **결과 해석**: 특정 감독들이 두 자릿수 작품을 보유하고 있습니다(다큐멘터리·스탠드업 코미디 연출자가 상위에 자주 보입니다). **빈 값을 먼저 제거**하지 않으면 `NaN` 이 가장 큰 집단으로 잡혀 결과가 왜곡되니 주의하세요.

<div style="margin:20px 0 4px;padding:14px 20px;border-radius:10px;background:#F4E7DF;border:1px solid #E7DCC6;">
<span style="font-size:19px;font-weight:800;color:#23211C;">✏️ 직접 풀어 보기 — 연습문제</span><br>
<span style="color:#5A5448;font-size:14px;line-height:1.7;">지금까지 쓴 도구(<code>value_counts</code>, 필터링, <code>groupby</code>, 정렬, 날짜 처리, 시각화)를 직접 써 봅니다. 모든 문제는 위에서 만든 <code>df</code> 를 그대로 씁니다. 앞쪽은 <b>빈칸(____) 채우기</b>, 뒤로 갈수록 <b>직접 작성</b>입니다. 막히면 위쪽 해당 분석 셀을 참고하세요.</span>
</div>

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 1 관람등급 Top 5</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>rating</code> 열(관람등급: TV-MA, PG-13 등)에서 <b>가장 많은 등급 상위 5개</b>를 구해 막대그래프로 그려 보세요.<br><b>힌트</b> &nbsp; 값별 개수를 세는 <code>value_counts</code> 로 집계 → 상위 몇 개만 <code>head</code> · 그래프는 <code>.plot(kind=?)</code> 로 종류를 고릅니다<br><b>예상</b> &nbsp; TV-MA 가 가장 많은, 오른쪽으로 갈수록 낮아지는 막대
</div>
</div>

In [ ]:
# ✏️ 연습문제 1 — 빈칸(____)을 채워 보세요
top_rating = df['rating'].____().head(5)   # 값별 개수를 세는 메서드
print(top_rating)

top_rating.plot(kind='____', color='#4C72B0', rot=0)   # 어떤 종류의 그래프일까요?
plt.title('관람등급 상위 5개'); plt.ylabel('개수'); plt.show()

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 2 2021년 영화만 세기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <b>2021년에 제작된 영화</b>만 필터링해 개수를 출력하세요. (연도는 <code>release_year</code>, 종류는 <code>type</code> 열)<br><b>힌트</b> &nbsp; 조건 두 개를 <code>&</code> 로 결합, 각 조건은 괄호로 감쌉니다 · 개수는 <code>len(...)</code><br><b>예상</b> &nbsp; 수백 편 규모의 정수 하나
</div>
</div>

In [ ]:
# ✏️ 연습문제 2 — 빈칸을 채워 보세요
movies_2021 = df[(df['release_year'] == ____) & (df['____'] == 'Movie')]   # 두 조건을 & 로 결합
print(f"2021년 제작 영화 수: {len(movies_2021)}")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 3 드라마 시즌 수 분포 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 드라마(<code>type</code> 이 'TV Show')만 골라 <b>시즌 수 분포</b>를 구하고, <i>시즌이 1개인 드라마</i>가 전체에서 차지하는 비중을 짐작해 보세요.<br><b>힌트</b> &nbsp; <code>duration</code> 은 <code>1 Season</code>·<code>2 Seasons</code> 형태 — 공백으로 나눠 맨 앞 숫자만 뽑고, <code>value_counts</code> 로 세어 봅니다<br><b>예상</b> &nbsp; 시즌 1개가 압도적으로 많다 (절반 이상)
</div>
</div>

In [ ]:
# ✏️ 직접 작성 — 드라마만 골라 시즌 수 분포를 구하고, 시즌 1개 비중을 출력
shows = ____                       # ← type 이 'TV Show' 인 행만 거르기
shows = shows.copy()
shows['seasons'] = ____            # ← duration 을 공백으로 나눠 맨 앞 숫자만
season_counts = ____               # ← seasons 값별 개수를 세고 숫자 순으로 정렬
print(season_counts)

ratio = ____ / len(shows) * 100    # ← 시즌이 '1' 인 개수를 전체로 나눠 비중(%)
print(f"\n시즌 1개인 드라마 비중: {ratio:.1f}%")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 4 국가별 평균 상영시간 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 7번에서 만든 <code>movies['duration_min']</code> 으로 <b>국가별 평균 영화 상영시간</b>을 구해 <b>긴 순서</b>로 상위 10개국을 출력하세요.<br><b>힌트</b> &nbsp; 나라별로 <code>groupby</code> 해 <code>duration_min</code> 평균 → <code>sort_values</code> 로 내림차순 정렬 · <code>country</code> 빈 값은 <code>dropna</code> 로 제외<br><b>예상</b> &nbsp; 영화 수가 적은 국가가 위로 올 수 있어 들쭉날쭉
</div>
</div>

In [ ]:
# ✏️ 직접 작성 — movies 의 duration_min 으로 국가별 평균 상영시간 상위 10개국 출력
subset = ____             # ← country 빈 값이 있는 행 제외
country_runtime = ____    # ← 국가별로 묶어 duration_min 의 평균 (groupby → mean)
country_runtime = ____    # ← 평균이 긴(내림차순) 순서로 정렬
country_runtime.head(10)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 5 연도별 추가량 추이 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 5번에서 날짜로 바꾼 <code>date_added</code> 에서 <b>연도</b>를 뽑아, <b>연도별 추가된 콘텐츠 수</b>를 선그래프로 그려 보세요.<br><b>힌트</b> &nbsp; <code>.dt.year</code> 로 연도 추출 → <code>value_counts</code> 후 연도 순으로 정렬(<code>sort_index</code>) · 빈 날짜는 <code>dropna</code> 로 제외<br><b>예상</b> &nbsp; 2010년대 중후반에 가파르게 오르는 선
</div>
</div>

In [ ]:
# ✏️ 직접 작성 — date_added 의 연도별 추가 콘텐츠 수를 선그래프로
dated = df.dropna(subset=['date_added']).copy()   # 빈 날짜 제외 (제공)
dated['year_added'] = ____     # ← date_added 에서 연도만 추출
added_per_year = ____          # ← 연도별 개수를 세고 연도 순으로 정렬
print(added_per_year)

added_per_year.plot(kind='____', marker='o', color='#C44E52')   # ← 선그래프 종류
plt.title('연도별 추가된 콘텐츠 수'); plt.xlabel('추가 연도'); plt.ylabel('콘텐츠 수'); plt.show()

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🎯 종합문제 드라마 장르 Top 10 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">종합</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>listed_in</code> 열에는 <code>Crime TV Shows, International TV Shows</code> 처럼 <b>장르가 콤마로 여러 개</b> 적혀 있습니다. 이번에는 <b>드라마(<code>type</code> 이 'TV Show')만</b> 골라, 장르를 콤마로 나눠 한 행에 하나씩 펼친 뒤 <b>가장 많은 장르 Top 10</b>을 가로 막대그래프로 그려 보세요.<br><b>힌트</b> &nbsp; 먼저 <code>type</code> 으로 드라마만 거른 뒤 → 장르 문자열을 콤마로 나누고(<code>split</code>) <code>explode</code> 로 한 행씩 펼쳐 → <code>value_counts</code> 로 상위 10개<br><b>예상</b> &nbsp; International TV Shows·TV Dramas 류가 상위에 오는 가로 막대
</div>
</div>

In [ ]:
# 🎯 종합문제 — 스켈레톤을 채워 완성해 보세요
# (조건) type 이 'TV Show' 인 콘텐츠만 대상으로 장르 Top 10
shows_only = ____      # ← type 이 'TV Show' 인 행만 거르기
genres = ____          # ← listed_in 을 ', ' 로 나눠 한 행에 하나씩 펼치기 (dropna → split → explode)
top_genres = ____      # ← 장르별 개수 상위 10개
print(top_genres)

top_genres.sort_values().plot(kind='____', color='#4C72B0')   # ← 가로 막대
plt.title('드라마 장르 Top 10 (TV Show)'); plt.xlabel('콘텐츠 수'); plt.tight_layout(); plt.show()

## 🔑 막히면 펼쳐보기 (힌트)

막히면 아래를 펼쳐 **어떤 메서드·형태**를 쓰는지 확인하세요. (완성 코드가 아니라 접근 방향입니다.)

<details>
<summary>연습문제 1 · 관람등급 Top 5</summary>

- 값별 개수는 `df['rating'].value_counts()`, 상위 5개는 뒤에 `.head(5)`
- 막대그래프는 `.plot(kind='bar', ...)`
</details>

<details>
<summary>연습문제 2 · 2021년 영화만 세기</summary>

- 두 조건 결합: `df[(조건1) & (조건2)]` — 각 조건은 괄호로 감쌉니다
- 조건 예: `df['release_year'] == 2021`, `df['type'] == 'Movie'`
- 개수는 `len(...)`
</details>

<details>
<summary>연습문제 3 · 드라마 시즌 수 분포</summary>

- 드라마만: `df[df['type'] == 'TV Show']`
- `'2 Seasons'` 에서 숫자만: `.str.split(' ').str[0]`
- 분포는 `.value_counts()`, 숫자 순 정렬은 `.sort_index(...)`
- 비중은 `(시즌 '1' 개수) / len(shows) * 100`
</details>

<details>
<summary>연습문제 4 · 국가별 평균 상영시간 Top 10</summary>

- 빈 국가 제외: `.dropna(subset=['country'])`
- 국가별 평균: `.groupby('country')['duration_min'].mean()`
- 긴 순서: `.sort_values(ascending=False)` → `.head(10)`
</details>

<details>
<summary>연습문제 5 · 연도별 추가량 추이</summary>

- 연도 추출: `date_added` 의 `.dt.year`
- 개수 집계·정렬: `.value_counts().sort_index()`
- 선그래프: `.plot(kind='line', marker='o')`
</details>

<details>
<summary>종합문제 · 드라마 장르 Top 10</summary>

- 드라마만: `df[df['type'] == 'TV Show']`
- 장르 펼치기: `listed_in` → `.dropna().str.split(', ').explode()`
- 상위 10개: `.value_counts().head(10)`
- 가로 막대: `.plot(kind='barh')`
</details>

## 12. 정리

이번 노트북에서 한 일을 돌아봅니다.

- **불러오기·구조 파악**: `read_csv` 로 표를 읽고, `shape`·`info`·`isna` 로 데이터의 크기와 빈 값을 점검했습니다.
- **세기**: `value_counts()` 로 종류·국가·등급·감독을 한 줄에 집계했습니다.
- **거르기**: `df[조건]` 불리언 인덱싱으로 원하는 행만 골랐고, `&`·`|` 로 조건을 결합했습니다.
- **묶기**: `groupby` 로 연도별·국가별 집계를 했습니다.
- **날짜·문자열 전처리**: `to_datetime`, `.dt` 접근자, `str.replace`·`str.split` 으로 실제 데이터를 분석 가능한 형태로 다듬었습니다.
- **시각화**: matplotlib·seaborn 으로 막대·선·히스토그램·영역 그래프를 그렸습니다.

3차시(`s3-03`)에서 순수 파이썬으로 여러 줄에 걸쳐 했던 일들 — 합계·필터·정렬·그룹 집계 — 이 여기서는 대부분 **한 줄**로 끝났습니다. 데이터가 커지고 분석이 복잡해질수록 pandas 의 이 간결함이 큰 힘이 됩니다.

> **데이터 분석의 흐름**은 늘 비슷합니다: **불러오기 → 구조 파악 → 결측치·전처리 → 질문별 집계·필터 → 시각화 → 해석**. 어떤 데이터셋을 만나도 이 순서를 떠올리면 막막함이 줄어듭니다.